## Word Representation and Semantics

In natural language processing (NLP), understanding a language involves multiple layers: morphology, syntax, and semantics. This module transitions into **semantics**—the meaning of words, sentences, and tokens. Because words can have different meanings based on context (e.g., "bank" of a river vs. financial "bank", or "apple" the fruit vs. the company), creating an effective, mathematically computable representation of word meaning is a core challenge in NLP.

### 1. Old-School Word Representation Techniques

Early NLP relied heavily on manual annotation and rigid encoding schemes, which suffered from severe scalability and semantic issues.

#### WordNet and Ontological Semantics

WordNet is a semi-automatically generated ontology where entities (nodes) are linked via relationships (holonym, meronym, synonym, antonym, hyponym).

* **Synset (Synonym Set):** Every specific sense (meaning) of a word is represented by a synset.
* **Gloss:** The manual description or definition of that synset (e.g., the gloss for one sense of "chump" is "a person who is gullible").
* **Drawbacks:**
1. Requires immense manual labor to curate.
2. Incomplete (misses contextual nuances, e.g., "proficient" vs. "good").
3. Difficult to update with newly coined words (e.g., "ninja", "nifty").
4. Highly subjective depending on the human annotator.
5. Primarily limited to nouns, verbs, adjectives, and adverbs.



#### One-Hot Encoding

A simple vectorization approach where a dictionary of size $V$ is created. Each unique word gets an ID. The word is represented by a vector of length $V$ where the position corresponding to its ID is $1$, and all other positions are $0$.

* **Drawbacks:**
1. **Orthogonality:** The dot product of any two different one-hot vectors is always $0$. It cannot capture semantic similarity (e.g., $Vector(motel) \cdot Vector(hotel) = 0$).
2. **Dimensionality:** As the dictionary size increases (e.g., to 1 million words), the vector size becomes impractically large and sparse.



---

### 2. Distributional Semantics and Count-Based Models

To solve the manual labor and orthogonality issues, modern word representation is built on **Distributional Semantics**.

> *"You shall know a word by the company it keeps." — J.R. Firth (1957)*

Instead of manual dictionaries, we look at the neighboring words (context) of a target word across large document corpora to guess its meaning.

#### The Term-Context Matrix

A matrix where rows represent individual terms (words) and columns represent context tokens. We define a **context window** (e.g., size 10) and count how many times a context word appears near a target word.

* **Row vector:** Represents the word as a target.
* **Column vector:** Represents the word as a context.
* Similar words (e.g., "apricot" and "pineapple") will have similar vector representations because they co-occur with similar context words (e.g., "sugar", "pinch").

#### TF-IDF (Term Frequency - Inverse Document Frequency)

Raw counts heavily bias frequently occurring stop words (like "the", "a", "of"). TF-IDF normalizes this.

* **Term Frequency (TF):** Measures how often term $t$ appears in document $d$. To dampen the effect of raw counts, we use log smoothing:
$TF(t, d) = \log(count(t, d)) + 1$
* **Document Frequency (DF):** The number of documents in which the term appears.
* **Inverse Document Frequency (IDF):** Penalizes words that appear in almost all documents (like stop words). Where $N$ is the total number of documents:
$IDF(t) = \log(\frac{N}{DF})$ *(Note: $1$ is often added to the denominator to prevent zero-division errors).*
* **Final Score:** $TF \times IDF$.

#### Drawbacks of Count-Based Approaches

1. **Sparsity & Size:** Matrices grow quadratically with vocabulary size and contain mostly zeros.
2. **Computational Cost:** Matrix decomposition techniques like SVD (Singular Value Decomposition) can create dense vectors but possess a quadratic time complexity $O(N^2)$, making it slow to update with new words.
3. **Bag-of-Words Limitation:** Ignores the exact ordering of words within the context window.

---

### 3. Prediction-Based Models: Word2Vec

Introduced by Tomas Mikolov and Jeff Dean at Google (2013), **Word2Vec** produces **dense vectors** (e.g., 100-500 dimensions). Dense vectors prevent overfitting, generalize better, and naturally capture synonymous relationships (if $W_1$ and $W_2$ both frequently co-occur with "car", Word2Vec infers $W_1$ and $W_2$ are similar).

Word2Vec features two primary architectures:

1. **CBOW (Continuous Bag of Words):** Uses the surrounding context words to predict the central target word.
2. **Skip-gram:** Uses the central target word to predict the surrounding context words. *(Highly popular, focused on in the lecture).*

#### Skip-Gram with Negative Sampling (SGNS)

SGNS frames word representation as a **self-supervised binary logistic regression** task. It classifies whether a given pair of words is a true (positive) context pair or a fake (negative) context pair.

**1. Generating Training Data:**

* **Positive Instances:** For a target word $W_t$, we slide a context window (e.g., $\pm 3$ words). The pairs $(W_t, W_{t-3}), (W_t, W_{t+1})$, etc., are marked as positive context pairs ($C_{pos}$).
* **Negative Instances:** We sample words outside the context window. For every 1 positive sample, we typically draw $K$ negative samples ($C_{neg}$) (e.g., $K=4$ or $5$). Taking more negative samples provides stronger statistical evidence and stabilizes training.

**2. The Logistic Regression Classifier:**
Given a target word $W$ and a context word $C$, we measure similarity using the dot product, converted to a probability using the **sigmoid function**:
$\sigma(x) = \frac{1}{1 + e^{-x}}$

* Probability that the pair is positive: $P(pos\vert{}W, C_{pos}) = \sigma(C_{pos} \cdot W)$
* Probability that the pair is negative: $1 - P(pos\vert{}W, C_{neg}) = 1 - \sigma(C_{neg} \cdot W) = \sigma(-C_{neg} \cdot W)$

**3. The Objective Function:**
We want to maximize the probability of true pairs and maximize the probability of negative pairs being classified as negative. We use Log-Likelihood:
$\text{Maximize } \log \sigma(C_{pos} \cdot W) + \sum_{i=1}^{K} \log \sigma(-C_{neg_i} \cdot W)$

To optimize, we take the derivative of this loss function with respect to the embedding parameters. Because the derivative of the sigmoid function is $\sigma(x)(1 - \sigma(x))$, the gradient updates simplify elegantly during gradient descent.

**4. The Embedding Matrix:**
We are training a matrix of size $2V \times D$ (where $V$ is vocabulary size and $D$ is vector dimension).

* The top half ($V \times D$) stores embeddings when the word acts as a **target** ($W$).
* The bottom half ($V \times D$) stores embeddings when the word acts as a **context** ($C$).
* During training, positive contexts are mathematically "pulled closer" to the target word in the vector space, while negative contexts are "pushed away".
* **Final Word Vector:** Usually obtained by summing the target and context vectors for a specific word.

---

### 4. Crucial Word2Vec Optimization Tricks

To make Word2Vec highly efficient and accurate, two sampling tricks are employed:

**1. Subsampling Frequent Words:**
Highly frequent words (like "the") provide little semantic value. Word2Vec removes words from the training set based on a probability formula. If a word's frequency $z(w)$ is very low (e.g., $<0.26\%$), it is kept with $100\%$ probability. If it appears too frequently, it has a high chance of being discarded, speeding up training and improving context quality.

**2. Modified Unigram Distribution for Negative Sampling:**
If negative samples were drawn based on pure unigram probability $P(w)$, stop words would dominate the negative samples. Word2Vec smooths this by raising the frequency to the power of $\alpha = 0.75$:
$P(w_i) = \frac{count(w_i)^{0.75}}{\sum count(w_j)^{0.75}}$
*Why 0.75?* It slightly decreases the sampling probability of highly frequent words (e.g., dropping from $0.99$ to $0.97$) and boosts the probability of rare words (e.g., increasing from $0.01$ to $0.03$), ensuring the model learns to push away from a diverse set of negative examples.

---

### 5. Evaluating Word2Vec & Handling Limitations

**Evaluation via Linear Analogies:**
A well-trained Word2Vec model captures relational semantics that can be evaluated using vector arithmetic.
*Example:* $Vector(King) - Vector(Man) + Vector(Woman) \approx Vector(Queen)$

**Limitations of Word2Vec:**

1. **Static Embeddings:** Every word gets exactly one vector. It cannot differentiate between the financial "bank" and the river "bank".
2. **Out-of-Vocabulary (OOV) Problem:** If a new word (e.g., "nifty") appears in the test set that wasn't in the training dictionary, the model has no embedding for it.

#### Subword Embeddings (FastText)

To solve the OOV problem, Facebook proposed **FastText**.
Instead of defining the vocabulary as whole words, the vocabulary consists of **character n-grams**.

* For the word `where` (with $n=3$), the subwords are: `<wh`, `whe`, `her`, `ere`, `re>`, and the special full word token `<where>`. (Angle brackets denote start/end of the word).
* Skip-gram is run on these subwords.
* If an unknown word like "nifty" appears during testing, its embedding is generated on the fly by **summing the trained embeddings of its subwords** (e.g., `<ni`, `nif`, `ift`, etc.).

---

### Practice Questions

**Q1. Which of the following is the primary mathematical reason one-hot encoding fails to capture semantic similarity between synonymous words like "hotel" and "motel"?**
A) The vectors are too dense.
B) The dot product of their respective vectors is always 0.
C) The vector dimensionality is too small for large vocabularies.
D) One-hot encoding assigns the same vector to synonymous words.
*Answer: B. One-hot vectors are strictly orthogonal; the dot product of two distinct one-hot vectors is always 0, meaning the model reads them as having zero similarity.*

**Q2. In the context of Word2Vec architectures, what is the fundamental difference between CBOW and Skip-gram?**
A) CBOW predicts context words given a central target word, while Skip-gram predicts the central target word given context words.
B) CBOW uses a count-based matrix, while Skip-gram uses a prediction-based neural network.
C) Skip-gram predicts context words given a central target word, while CBOW predicts the central target word given context words.
D) Skip-gram operates on character n-grams, while CBOW operates on whole words.
*Answer: C. CBOW looks at the surrounding bag of words to guess the center word, whereas Skip-gram takes the center word and tries to predict its surrounding context.*

**Q3. When performing Negative Sampling in Word2Vec, why is the unigram probability of words raised to the power of $\alpha = 0.75$?**
A) To completely eliminate stop words from being sampled.
B) To decrease the probability of sampling rare words and increase the probability of frequent words.
C) To satisfy the requirements of the sigmoid derivative function.
D) To slightly boost the sampling probability of rare words while dampening the dominance of highly frequent words.
*Answer: D. Raising a small fraction to a power less than 1 increases its value relative to larger fractions, ensuring rare words are sampled more often than raw counts would dictate.*

**Q4. The Skip-gram with Negative Sampling (SGNS) model optimizes its parameters by functioning as which type of machine learning model?**
A) A multi-class Support Vector Machine.
B) A binary logistic regression classifier.
C) A recurrent neural network.
D) An unsupervised Singular Value Decomposition model.
*Answer: B. SGNS treats context prediction as a binary classification problem (predicting whether a word pair is a true context pair or a randomly sampled negative pair) using a sigmoid function and log-likelihood.*

**Q5. How does FastText overcome the Out-of-Vocabulary (OOV) problem inherent in standard Word2Vec?**
A) It queries the WordNet ontology whenever an unknown word is encountered.
B) It assigns a random dense vector to any new word it encounters during testing.
C) It represents words as a sum of their constituent character n-gram embeddings.
D) It re-trains the entire corpus using SVD when a new word is found.
*Answer: C. Because FastText learns embeddings for subword components (n-grams), it can construct an embedding for an unseen word by summing the vectors of its known character n-grams.*

---

### Key Takeaways for the Exam

* **WordNet:** Manual ontology, high labor, incomplete coverage.
* **One-Hot Vectors:** Highly dimensional, orthogonal (zero dot product for distinct words).
* **TF-IDF:** Down-weights common words. Formula: $TF = \log(count) + 1$, $IDF = \log(N/DF)$. Count matrices suffer from sparsity and $O(N^2)$ decomposition costs.
* **Word2Vec (Prediction-based):** Generates dense vectors. Solves sparsity and orthogonality. Allows linear semantic math ($King - Man + Woman = Queen$).
* **CBOW vs Skip-Gram:** CBOW (Context $\rightarrow$ Target). Skip-Gram (Target $\rightarrow$ Context).
* **SGNS Mechanics:** Classifies positive vs. negative pairs. Sigmoid activation function. Loss is based on negative log-likelihood. Learns two vectors per word (Target matrix and Context matrix).
* **Word2Vec Tricks:** Subsampling drops highly frequent words. Negative sampling uses $\alpha=0.75$ to ensure rare words are utilized effectively as negative examples.
* **Word2Vec Weaknesses:** Static embeddings (no context-specific senses) and fails on OOV (unseen) words.
* **FastText:** Uses character n-grams to create subword embeddings, solving the OOV problem by summing n-grams for unseen words.